# Job Arrays with Map

This notebook demonstrates how to use `.map()` to submit multiple tasks as SLURM job arrays.

You'll learn how to:
- Map a task over a list of inputs
- Map with multiple parameters
- Inspect array future properties
- Control chunking with `max_array_size`

## Setup

In [ ]:
from prefect import flow, task
from prefect_submitit import SlurmTaskRunner

## Define Tasks

In [ ]:
@task
def square(x: int) -> int:
    return x * x


@task
def add(a: int, b: int) -> int:
    return a + b

## Basic Map

`.map()` submits one task per input element. On SLURM, these become a job array.

In [ ]:
@flow(task_runner=SlurmTaskRunner(execution_mode="local"))
def basic_map_flow():
    futures = square.map(x=[1, 2, 3, 4, 5])
    results = [f.result() for f in futures]
    print(f"Squares: {results}")
    return results


results = basic_map_flow()
assert results == [1, 4, 9, 16, 25]

## Multi-Parameter Map

When a task takes multiple arguments, pass a list for each parameter. They are zipped element-wise.

In [ ]:
@flow(task_runner=SlurmTaskRunner(execution_mode="local"))
def multi_param_map_flow():
    futures = add.map(a=[1, 2, 3], b=[10, 20, 30])
    results = [f.result() for f in futures]
    print(f"Sums: {results}")
    return results


results = multi_param_map_flow()
assert results == [11, 22, 33]

## Inspect Array Futures

Each future from `.map()` is a `SlurmArrayPrefectFuture` with array-specific properties.

In [ ]:
@flow(task_runner=SlurmTaskRunner(execution_mode="local"))
def inspect_array_flow():
    futures = square.map(x=[10, 20, 30])

    for i, f in enumerate(futures):
        result = f.result()
        print(f"Task {i}:")
        print(f"  slurm_job_id: {f.slurm_job_id}")
        print(f"  array_job_id: {f.array_job_id}")
        print(f"  array_task_index: {f.array_task_index}")
        print(f"  result: {result}")


inspect_array_flow()

## Chunking with `max_array_size`

SLURM clusters have a maximum array size (often 1000). If you submit more items than
the limit, `prefect-submitit` automatically splits them into multiple job arrays.

Here we set `max_array_size=3` to demonstrate auto-chunking with a small example.

In [ ]:
runner = SlurmTaskRunner(execution_mode="local", max_array_size=3)


@flow(task_runner=runner)
def chunked_map_flow():
    # 7 items with max_array_size=3 -> multiple array submissions
    futures = square.map(x=[10, 20, 30, 40, 50, 60, 70])
    results = [f.result() for f in futures]
    print(f"Results: {results}")

    # Show that different chunks have different array_job_ids
    job_ids = {f.array_job_id for f in futures}
    print(f"Number of array submissions: {len(job_ids)}")

    return results


results = chunked_map_flow()
assert results == [100, 400, 900, 1600, 2500, 3600, 4900]